---
### **Robustness Test**

`CSI_strategy.ipynb`의 본 전략을 같은 데이터·같은 파라미터·같은 엔진으로 다시 만들어 여러 각도에서 검증한다.

**본 전략**: 시총 상위 30% → 롱: 실제로 주식수가 줄어든 종목(CSI 하위, 최대 30) / 숏: CSI 상위 30종목 → 롱 감소규모×점진성(k=1.0) / 숏 저변동성(k=0.5) 가중
→ AUM 1000억을 port_A·port_B·port_C(각 333.3억)로 나눠 한 달씩 어긋나게 3개월 주기 리밸런싱 → 거래대금 제약 없이 배당반영 수정주가를 체결·평가 기준가격으로 사용
→ 왕복 50bp(실제 매매일 부과) + 대차 연 3%, 월말 배당반영 수정주가 평가. 성과 측정은 2012-02-29 ~ 2026-08-31.

모든 성과·그래프는 본 전략(거래대금 미포함) 기준이다.

1. 분위수 검증: CSI 5분위 포트폴리오 성과가 단조적인지
2. 팩터 회귀: 4-Factor, 4-Factor + CSI 팩터 (거래대금 미포함)
3. 이중정렬: 규모·모멘텀으로 통제한 CSI 5분위 Sharpe (5x5 히트맵)
4. 파라미터 민감도: 종목 수 / 가중 방식 / k값(표본내) / 거래비용 / 관측기간(τ) / 체결 기준 / 시작 시점 / 롱 후보 선정 기준 (가능한 곳은 표본내·표본외를 나눠서)
5. 표본내(2012.03~2020.12) / 표본외(2021.01~2026.08) 성과
6. 체결률: 포트폴리오(333.3억)마다 목표비중을 실제로 얼마나 채우는지

---
## **0. 데이터·팩터·파라미터·엔진 (전략 노트북과 같은 코드)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

##### **데이터 로드** — 수정주가(배당 반영·미반영), 시가총액, 거래대금

In [ ]:
adj_close             = pd.read_csv("Factor Construction input/코스피코스닥_배당반영수정주가.csv", index_col=0, parse_dates=True)   # 배당반영 수정주가
adj_close_nodiv       = pd.read_csv("Factor Construction input/코스피코스닥_수정주가.csv", index_col=0, parse_dates=True)         # 배당 미반영 수정주가
market_capitalization = pd.read_csv("Factor Construction input/코스피코스닥_시가총액.csv", index_col=0, parse_dates=True)

return_monthly     = adj_close_nodiv.resample('ME').last().pct_change(fill_method=None)                   # 월별 주가수익률 (배당 미반영, CSI용)
market_cap_monthly = market_capitalization[market_capitalization.index.notna()].resample('ME').last()   # 월말 시가총액
# 로그가 정의되지 않는 시가총액 0 이하와 수익률 -100% 이하 관측치는 NaN으로 제외한다.
market_cap_for_log = market_cap_monthly.where(market_cap_monthly > 0)
return_for_log     = return_monthly.where(return_monthly > -1)

print(f"종목 수 — 주가 {adj_close.shape[1]:,} / 시가총액 {market_capitalization.shape[1]:,}")
print(f"주가 기간 — {adj_close.index.min().date()} ~ {adj_close.index.max().date()}")

##### **팩터·KOSPI 월간 수익률** — 2장 회귀(MKT·SMB·HML·MOM·RF)와 5장 KOSPI 비교용

In [ ]:
factor_raw = pd.read_csv("Performance Evalution input/factor_monthly.csv", index_col=0, parse_dates=True)
factor_raw.columns = ['HML', 'SMB', 'MOM', 'KOSPI', 'RF']
factors_monthly = factor_raw[['KOSPI', 'SMB', 'HML', 'MOM']].pct_change(fill_method=None)
factors_monthly['RF'] = factor_raw['RF'].astype(float) / 100 / 12
factors_monthly = factors_monthly.dropna()
factors_monthly.index = factors_monthly.index + pd.offsets.MonthEnd(0)   # 월 마지막 영업일 -> 달력 월말(전략 수익률 인덱스와 맞춤)

##### **CSI 팩터** — 4장 관측기간(τ) 민감도에서 `build_csi_factor(τ)`를 다시 쓴다

In [ ]:
# 관측기간 tau에 맞춰 가격 효과를 제거한 CSI(주식수 변화) 팩터를 생성한다.
def build_csi_factor(tau):
    """τ개월 로그 시가총액 변화 − 같은 기간 누적 로그 주가수익률(배당 미반영) = 주식수 변화율(iota)."""
    log_me         = np.log(market_cap_for_log)                  # 0 이하를 제외한 로그 시가총액
    cum_log_return = np.log1p(return_for_log).rolling(tau).sum() # -100% 이하를 제외한 누적 로그수익률
    return (log_me - log_me.shift(tau)) - cum_log_return   # 가격효과를 뺀 주식수 변화: 작을수록 롱, 클수록 숏


TAU = 48   # 관측기간(개월)
csi_factor = build_csi_factor(TAU)
csi_factor['삼성전자'].dropna().tail()   # 예시: 삼성전자 (자사주 소각으로 음수)

##### **파라미터**

In [ ]:
# ---- 3-1. 유니버스·종목 선정 ----
n                  = 30        # 레그별 최대 종목 수 (롱: 실제 감소 종목 전부, 최대 30 / 숏: 30)
CAP_PCT            = 0.70      # 유니버스: 시가총액 상위 30%

# ---- 3-2. 가중 방식 ----
LONG_WEIGHT_K      = 1.0       # 롱: 감소규모 x 점진성 순위 가중 (0~2배)
SHORT_WEIGHT_K     = 0.5       # 숏: 60거래일 저변동성 순위 가중 (0.5~1.5배)
REAL_CHANGE_EPS    = 0.005     # CSI < -0.5%인 종목만 '실제 감소' 종목으로 보고 롱 가중을 적용
MONTH_EPS          = 0.001     # 점진성: 월 주식수 변화가 -0.1%보다 작은 달만 '감소한 달'로 센다(시가총액 반올림 노이즈 제외)

# ---- 3-3. 3등분 스태거링 ----
AUM                = 100_000_000_000   # 1000억 (롱 100% + 숏 100%)
N_PORTS            = 3                 # port_A·port_B·port_C
AUM_PORT           = AUM / N_PORTS     # 포트폴리오당 333.3억

# ---- 3-4. 거래대금 미포함 체결 ----
PARTICIPATION_RATE = None      # 거래대금 변수를 사용하지 않음
MAX_EXECUTION_DAYS = 10        # 편입 창: 리밸런싱일부터 10영업일 (못 채운 몫은 현금)
EXIT_UNLIMITED     = True      # 정리(매도)는 구간 끝까지 계속

# ---- 3-5. 거래비용·수익률 ----
SPREAD             = 0.0050    # 왕복 50bp (매수·매도 각 25bp)
BORROW_ANNUAL      = 0.03      # 숏 대차수수료 연 3%
WINSORIZE_CAP      = 20.0      # 종목 수익률 +2000% 초과분은 데이터 오류로 보고 캡

# ---- 백테스트 계산 기준 ----
# 1) 체결시점: 월말 데이터로 신호를 만든 뒤 같은 리밸런싱일에 체결한다.
# 2) 고정자본: 초기자본 AUM=1000억, port_A·B·C는 각각 AUM_PORT(약 333.3억)를 계속 사용한다.
#    현금과 운용수익을 다음 리밸런싱의 원금에 재투자하지 않고 매 구간 목표비중 1.0으로 다시 맞춘다.
# 3) 대차비용: 실제 일별 숏 잔고·종목별 대차료 대신 달성 숏 비중에 연 3%를 경과일수로 안분한다.
# 4) 청산/현금: 구간 안의 매도분은 cash에 더해 같은 구간 매수여력에 반영한다.
#    다만 다음 구간은 다시 고정 AUM 기준으로 시작하므로 매매손익으로 총 운용규모를 키우지 않는다.
# 성과 NAV는 월별 수익률을 복리 누적한 지표이며, 체결금액 산정에 쓰는 기준 AUM은 항상 고정이다.


##### **리밸런싱 일정·성과 함수**

port_A·B·C 리밸런싱 일정은 전략 노트북과 같다. 성과는 세 포트폴리오가 모두 투자된 2012-02-29부터 잰다. `quarter_ends`는 팩터 회귀용 분기 격자다(2012-03-31부터, 마지막 구간은 2026-07~08의 2개월).

In [ ]:
start_point = '2011-12-31'
end_point   = '2026-08-31'

all_month_ends = pd.date_range(start=start_point, end=end_point, freq='ME')              # 월말 평가 시점
PORT_NAMES     = [f"port_{chr(ord('A') + k)}" for k in range(N_PORTS)]                  # port_A, port_B, port_C
port_schedules = [all_month_ends[k::N_PORTS] for k in range(N_PORTS)]                   # 포트폴리오별 리밸런싱일 (3개월 주기)
port_schedules = [s if s[-1] == all_month_ends[-1] else s.append(all_month_ends[-1:]) for s in port_schedules]

quarter_ends = pd.date_range(start='2012-03-31', end=end_point, freq='QE')
# 마지막 분석일이 정규 분기말이 아니면 최종 관측일을 별도 평가 시점으로 추가한다.
if quarter_ends[-1] < pd.Timestamp(end_point):
    quarter_ends = quarter_ends.append(pd.DatetimeIndex([end_point]))


# 월별 수익률 시계열을 전략 비교에 공통으로 쓰는 성과지표로 요약한다.
def perf(ret):
    """월별 수익률(첫 행 = 수익률 0인 시작점) -> CAGR / 변동성 / Sharpe(초과수익, RF=factor_monthly.csv) / MDD(월말 배당반영 수정주가 평가, 시작점 포함)."""
    r = ret.iloc[1:].dropna()
    ex = r - factors_monthly['RF'].reindex(r.index).ffill()   # 월별 초과수익률 (RF: factor_monthly.csv, 연율%/100/12)
    nav = (1 + ret.fillna(0)).cumprod()
    return {"CAGR": nav.iloc[-1] ** (12 / len(r)) - 1, "Vol": r.std() * np.sqrt(12),
            "Sharpe": ex.mean() / ex.std() * np.sqrt(12), "MDD": (nav / nav.cummax() - 1).min()}


# 월별 수익률을 회귀분석용 분기 격자에 맞춰 구간 복리수익률로 변환한다.
def _compound_to_grid(monthly, grid):
    """월별 수익률(Series/DataFrame)을 grid 구간별로 복리 환산한다(첫 행은 0)."""
    is_series = isinstance(monthly, pd.Series)
    frame = monthly.to_frame() if is_series else monthly
    out = pd.DataFrame(index=grid, columns=frame.columns, dtype=float)
    out.iloc[0] = 0.0
    for i in range(len(grid) - 1):
        seg = frame[(frame.index > grid[i]) & (frame.index <= grid[i + 1])]
        out.loc[grid[i + 1]] = (1 + seg).prod() - 1 if len(seg) else 0.0
    return out.iloc[:, 0] if is_series else out

##### **엔진 (전략 노트북 3-1 ~ 3-5와 같은 코드) + 분위수 포트폴리오 함수**

분위수 포트폴리오는 거래정지·데이터 부족 종목을 분위 계산 전에 빼고 동일가중으로 담는다(같은 port_A·B·C 일정·거래대금 미포함·월말 배당반영 수정주가 평가).

In [ ]:
# 3-1. 유니버스·거래 가능 종목·종목 선정
_liq_px_cache = {}
# 종목별 유효 가격을 캐시해 체결·평가 단계의 반복적인 결측 제거를 피한다.
def _liq_price_series(ticker):
    """종목별 유효 배당반영 수정주가 시계열을 캐시해 반복 조회 비용을 줄인다."""
    # 처음 조회하는 종목만 원본 가격에서 결측치를 제거해 캐시에 저장한다.
    if ticker not in _liq_px_cache:
        _liq_px_cache[ticker] = adj_close[ticker].dropna()
    return _liq_px_cache[ticker]


_daily_ret     = adj_close.pct_change(fill_method=None)  # 거래가능 필터와 숏 저변동성 가중의 기초 일수익률
_vol60_monthly = (_daily_ret.rolling(60, min_periods=40).std() * np.sqrt(252)).resample('ME').last()   # 60거래일 롤링 변동성(연환산), 월말 값


# 신호일 기준 가격 이력과 변동성이 충분해 실제 거래 가능한 종목인지 판별한다.
def _tradable(date, tickers):
    """거래정지·데이터 부족 종목 판정: 최근 60거래일 변동성이 0(가격이 안 움직임)이거나 계산 불가(관측치 40개 미만)면 거래 불가."""
    v = _vol60_monthly.loc[date].reindex(tickers)
    return (v.notna() & (v > 1e-8)).values


# 시가총액 유니버스 안에서 CSI 방향에 따라 롱·숏 후보와 우선순위를 정한다.
def select_candidates(date, P):
    """시총 상위 30% 유니버스에서 CSI 순위로 뽑는다.
    롱: 실제로 주식수가 줄어든(CSI < -0.5%) 종목만, CSI가 낮은 순서. 숏: CSI가 높은 순서. 두 레그 모두 앞에서 최대 n종목을 담는다."""
    cap_today = market_cap_monthly.loc[date].dropna()                         # 리밸런싱일 시가총액 관측 종목
    universe = cap_today[cap_today >= cap_today.quantile(P['cap_pct'])].index # 시총 상위 30% 유니버스
    ff = P['factor'].loc[date].reindex(universe).dropna()                    # 유니버스 안 CSI 유효값
    long_c = ff[ff < -REAL_CHANGE_EPS]                                      # CSI<-0.5% 실제 주식수 감소 필터 / 시총 변화가 소수점으로 변하는 경우 배제.
    long_ranked = list(long_c.sort_values(ascending=True, kind='mergesort').index) # 감소폭 큰 순서
    short_ranked = list(ff.sort_values(ascending=False, kind='mergesort').index)   # 증가폭 큰 순서
    return long_ranked, short_ranked
# 3-2. 가중 방식 (선정된 30종목 안에서 비중만 조정)
_d_monthly = np.log(market_cap_for_log).diff() - np.log1p(return_for_log)   # 로그 불능 관측치를 제외한 월별 주식수 로그변화


# 원점수 대신 횡단면 순위를 사용해 극단값의 영향을 제한한 목표가중 배수를 만든다.
def _rank_tilt(score, k):
    """score의 순위를 0~1로 펼친 뒤(꼴찌 0, 1등 1) 1 + k(2*순위 - 1) 가중으로 바꾼다 (k=1: 0~2배, k=0.5: 0.5~1.5배).
    점수가 같은 종목들은 그 순위들의 평균 순위를 같이 받는다(예: 꼴찌 10개가 동점이면 1~10위의 평균).
    score가 없는(데이터 부족) 종목은 중립 가중(1.0)을 준다 — 실제 체결 여부는 3-4 체결 단계에서 결정된다."""
    s = score.dropna()
    pr = (s.rank(method='average') - 1) / (len(s) - 1) if len(s) > 1 else s * 0 + 0.5
    return (1 + k * (2 * pr - 1)).clip(lower=0.02).reindex(score.index).fillna(1.0)


# 주식수 감소가 한 달에 집중됐는지 여러 달에 걸쳐 점진적으로 발생했는지 측정한다.
def _lump_dn(date, tickers, tau):
    """관측기간(τ) 감소분 중 가장 큰 한 달이 차지하는 비중 (1 = 한 달에 몰아서 감소, 0에 가까울수록 여러 달에 나눠 감소).
    월 주식수 변화가 -MONTH_EPS(-0.1%)보다 작은 달만 '감소한 달'로 센다. 시가총액 반올림 때문에 생기는 ±0.001% 수준의 노이즈를 걸러낸다."""
    loc = _d_monthly.index.get_loc(date)
    w = _d_monthly.iloc[loc - tau + 1: loc + 1].reindex(columns=tickers)
    neg = (-w).where(w < -MONTH_EPS, 0.0)
    return (neg.max() / neg.sum().replace(0, np.nan)).fillna(1.0)


# 선택된 롱 종목에 가중 방식별 목표비중을 배분한다.
def _long_weight_target(date, top_n, P):
    """롱 목표비중(합 1). 'strategy': 감소규모 x 점진성 순위로 가중(k=long_k)."""
    # 동일가중 실험에서는 신호 강도와 무관하게 종목 수로 균등 배분한다.
    if P['weight_mode'] == 'equal':
        return {t: 1.0 / len(top_n) for t in top_n}
    # 시가총액가중 실험에서는 리밸런싱일 시총 비율을 그대로 목표비중으로 쓴다.
    if P['weight_mode'] == 'cap':
        cap = market_cap_monthly.loc[date, top_n]
        return (cap / cap.sum()).to_dict()
    c = P['factor'].loc[date]
    real = [t for t in top_n if c[t] < -REAL_CHANGE_EPS]
    w = pd.Series(1.0, index=top_n)
    # 순위 비교가 가능한 실제 감소 종목이 2개 이상일 때만 전략 가중을 적용한다.
    if len(real) >= 2:
        score = (-c.reindex(real)) * (1 - _lump_dn(date, real, P['tau'])) # 감소규모×점진성
        tw = _rank_tilt(score, P['long_k'])
        w.loc[real] = tw / tw.sum() * len(real)
    return (w / w.sum()).to_dict()


# 선택된 숏 종목에 가중 방식별 목표비중을 배분하고 고변동 종목 비중을 낮춘다.
def _short_weight_target(date, top_n, P):
    """숏 목표비중(합 1). 'strategy': 60거래일 변동성이 낮을수록 가중(k=short_k). 고변동성(숏스퀴즈 위험) 종목의 비중을 줄인다."""
    # 동일가중 실험에서는 모든 숏 종목에 같은 목표비중을 부여한다.
    if P['weight_mode'] == 'equal':
        return {t: 1.0 / len(top_n) for t in top_n}
    # 시가총액가중 실험에서는 리밸런싱일 시총 비율을 목표비중으로 사용한다.
    if P['weight_mode'] == 'cap':
        cap = market_cap_monthly.loc[date, top_n]
        return (cap / cap.sum()).to_dict()
    v = _vol60_monthly.loc[date].reindex(top_n) # 숏스퀴즈 완화를 위한 60일 변동성
    w = _rank_tilt(-v, P['short_k'])
    return (w / w.sum()).to_dict()
# 3-4. 거래대금 미포함 체결 (레그 하나, 리밸런싱 구간 하나)
# 리밸런싱일 이후 유효 가격이 남아 있어 포지션을 계속 평가할 수 있는지 확인한다.
# [상장폐지 종목 처리 점검: 수정하지 않는 근거]
# _liq_alive는 전체 데이터에서 종목의 마지막 가격일(px.index[-1])을 보므로, 형식상 리밸런싱일(day0) 이후 정보를 쓴다.
# 다만 실제로 판정에 쓰이는 정보는 day0 시점에 이미 공개된 사실이라, 성과에 미래 정보가 들어가지 않는다고 판단해 그대로 둔다.
#
# 1) False가 나온 경우: 전체 백테스트에서 42회(15종목). 모두 마지막 거래일이 day0보다 7~86일 앞선다.
#    → day0에는 이미 상장폐지가 끝나 공개된 상태였다(거래정지 중 재개 여부를 미리 안 경우는 없음).
#
# 2) 종목 유형:
#    - 합병·공개매수 후 자진 상장폐지(우리금융지주, 현대하이스코, 오스템임플란트, 락앤락, 커넥트웨이브 등 약 10종목):
#      주주는 합병 신주나 공개매수 대금으로 대략 마지막 가격만큼 받으므로, 마지막 가격 청산이 현실에 가깝다.
#    - 부실 상장폐지(쌍용건설, 이아이디 등 5종목): 정리매매 폭락(마지막 20거래일 -83~-99%)이 이미 가격 데이터에 들어 있어
#      손실이 반영된 뒤이고, 남은 비중도 포트폴리오의 0.01~0.1% 수준이다.
#
# 3) 극단 가정으로 다시 돌린 결과 (거래대금 제약 없음, 표본외 2021.01~2026.08):
#    - 현재 코드(마지막 가격 청산):                         표본외 Sharpe 1.87, CAGR 27.3%
#    - 미래 정보 없이 영원히 보유(마지막 가격 고정, 자금 묶임):   표본외 Sharpe 1.76, CAGR 23.3%
#    - 폐지된 롱 종목 -100% 손실:                           표본외 Sharpe 1.72, CAGR 25.6%
#    두 대안 모두 현실보다 가혹한 하한이고, 이 경우에도 표본외 Sharpe는 1.7대로 결론이 바뀌지 않는다.
#
# 4) 거래정지 후 재개된 종목은 True로 계속 보유(정지 기간 매도 불가)되어 현실과 같다.
# 월말 평가에서 한 구간 내내 가격이 없는 보유 종목을 빠뜨리는지도 따로 확인했다: 보정해도 결과 동일(해당 사례 0건).
# 엄밀하게 하려면 KRX 상장폐지일(당시 공시 정보)로 판정해야 하지만, 위 결과상 성과 수치에는 영향이 없다고 본다.
def _liq_alive(ticker, day0):
    """상장폐지 등으로 리밸런싱일 이후 가격이 없는 종목이면 False (잔여 포지션은 마지막 가격에 청산된 것으로 본다)."""
    px = _liq_price_series(ticker)
    return len(px) > 0 and px.index[-1] >= day0 #


# 리밸런싱 구간 안에서 월별 수익률을 평가할 체크포인트 목록을 만든다.
def _checkpoints(start_date, end_date, calendar):
    """구간 안의 월말 평가 시점 (마지막은 구간 종료일)."""
    cps = [d for d in calendar if start_date < d <= end_date]
    # 정규 월말 목록에 구간 종료일이 없더라도 마지막 가치평가는 반드시 포함한다.
    if not cps or cps[-1] != end_date:
        cps = cps + [end_date]
    return cps


# 한 레그의 기존 보유분 청산, 신규 편입, 월말 평가를 일별 가격 기준으로 시뮬레이션한다.
def _liq_leg_step_ckpt(prev_w, ranked_candidates, start_date, end_date, n, target_total, top_targets, checkpoint_dates,
                       aum, participation, max_days):
    """레그(롱 또는 숏) 하나를 한 리밸런싱 구간 동안 하루 단위로 체결한다.
    - 거래대금 한도는 사용하지 않으며, 가격이 있는 날 목표 수량을 전량 체결한다.
    - 편입: 리밸런싱일부터 max_days 영업일 동안 상위 n종목을 목표비중까지 매수. 못 채운 몫은 현금(후순위로 넘기지 않음)
    - 정리: 후보에서 빠진 종전 보유 종목은 구간 끝까지 같은 한도로 매도. 못 판 종목만 다음 구간으로 이월
    반환: (기여수익률, 다음 구간 시작 비중, 거래 비중 합, 보유 비중 합, 편입 후 종목별 비중,
          월말 평가 값, 월말까지 누적 거래 비중)"""
    D = max_days
    base = target_total / n
    top_targets = top_targets or {}
    # 종목별 명시 목표비중이 없을 때 사용할 기본 동일가중 목표를 반환한다.
    def tgt(t):
        """종목별 목표비중을 반환하고 별도 목표가 없으면 동일가중을 쓴다."""
        return top_targets.get(t, base)
    cal = adj_close.index
    pre = cal[cal <= start_date]
    day0 = pre[-1] if len(pre) else start_date  # 월말 신호를 같은 리밸런싱일의 배당반영 수정주가로 체결
    days = cal[(cal >= day0) & (cal <= end_date)]
    ranked = list(ranked_candidates)

    hold = {t: float(w) for t, w in prev_w.items() if abs(w) > 1e-12 and _liq_alive(t, day0)}
    w0 = dict(hold)
    buys, sells = {}, {}
    px_d, cap_d = {}, {}

    # 종목별 구간 가격과 일별 주문 한도를 최초 한 번만 준비한다.
    def prep(t):
        """종목의 구간 가격과 일별 체결한도를 한 번 계산해 캐시한다."""
        # 아직 준비되지 않은 종목만 배열을 만들어 반복 계산을 방지한다.
        if t not in px_d:
            px_d[t] = _liq_price_series(t).reindex(days).values
            # 거래대금 제약을 끈 본 설정에서는 유효 가격이 있는 날 전량 체결할 수 있다.
            if participation is None:
                cap_d[t] = np.full(len(days), np.inf)
                return
            cap_d[t] = np.full(len(days), np.inf)

    for t in list(hold) + ranked[:n]:
        prep(t)

    # 해당 거래일에 체결 가능한 유효 양수 가격만 반환한다.
    def price(t, k):
        """구간 k번째 날의 유효한 양수 배당반영 수정주가만 반환한다."""
        p = px_d[t][k]
        return p if (p == p and p > 0) else None

    group, selected = ranked[:n], set(ranked[:n])
    cash = target_total - sum(hold.values()) # 고정 목표금액(비중 1.0)에서 기존 보유분을 뺀 구간 내 현금

    k1 = len(days) if EXIT_UNLIMITED else min(D, len(days))
    for k in range(0, k1):
        room = {}
        for t in set(group) | set(hold):
            prep(t)
            room[t] = cap_d[t][k] if price(t, k) is not None else 0.0
        for t in list(hold):                                # 매도: 후보에서 빠진 종목 / 목표 초과분
            keep = tgt(t) if t in selected else 0.0
            # 편입 창이 끝난 뒤에도 선정 종목이면 추가 리밸런싱 없이 기존 비중을 유지한다.
            if k >= D and t in selected:
                continue
            over = hold[t] - keep
            sold_so_far = sum(x for x, _, _ in sells.get(t, []))
            left_w0 = w0.get(t, 0.0) - sold_so_far
            # 목표 초과분, 당일 체결 여력, 실제 기존 보유분이 모두 있을 때만 매도한다.
            if over > 1e-12 and room.get(t, 0.0) > 1e-12 and left_w0 > 1e-12:
                sq = min(over, room[t], left_w0)
                # 부동소수점 오차 수준의 미세 주문은 거래로 기록하지 않는다.
                if sq > 1e-12:
                    sells.setdefault(t, []).append((sq, price(t, k), k))
                    hold[t] -= sq; room[t] -= sq; cash += sq # 매도분은 같은 구간의 매수 가능금액에 반영
        for t in (group if k < D else []):                   # 매수: 편입 창(D영업일) 안에서만
            need = max(tgt(t) - hold.get(t, 0.0), 0.0)
            q = min(room.get(t, 0.0), need, max(cash, 0.0))
            # 목표비중과 현금 여력을 반영한 주문량이 유의미할 때만 매수한다.
            if q > 1e-12:
                buys.setdefault(t, []).append((q, price(t, k), k))
                hold[t] = hold.get(t, 0.0) + q; room[t] -= q; cash -= q

    contrib_total, traded_total, achieved_total = 0.0, 0.0, 0.0
    end_vals, p_start_map = {}, {}
    for t in set(hold) | set(buys) | set(sells):
        px = _liq_price_series(t)
        before = px.loc[:start_date]
        during = px[(px.index > start_date) & (px.index <= end_date)]
        has_start_price = len(before) > 0 and pd.notna(before.iloc[-1]) and before.iloc[-1] > 0
        has_end_price = len(during) > 0 and pd.notna(during.iloc[-1]) and during.iloc[-1] > 0
        p_start_map[t] = before.iloc[-1] if has_start_price else np.nan
        sold = sum(x for x, _, _ in sells.get(t, []))
        keep = w0.get(t, 0.0) - sold
        # 구간 안에 시작·종료 평가가격이 모두 있을 때만 실제 가격수익률을 반영한다.
        if has_start_price and has_end_price:
            p_start, p_end = before.iloc[-1], during.iloc[-1]
            held_ret = min(p_end / p_start - 1, WINSORIZE_CAP)
            contrib = keep * held_ret
            contrib += sum(x * min(p_sell / p_start - 1, WINSORIZE_CAP) for x, p_sell, _ in sells.get(t, []))
            val = keep * (1 + held_ret)
            for x, p_buy, _ in buys.get(t, []):
                buy_ret = min(p_end / p_buy - 1, WINSORIZE_CAP)
                contrib += x * buy_ret
                val += x * (1 + buy_ret)
        else:
            # 구간 중 새로운 유효가격이 없으면 해당 종목의 수익률을 0%로 처리한다.
            contrib, val = 0.0, keep + sum(x for x, _, _ in buys.get(t, []))
        contrib_total += contrib
        end_vals[t] = val
        traded_total += sold + sum(x for x, _, _ in buys.get(t, []))
        achieved_total += keep + sum(x for x, _, _ in buys.get(t, []))

    end_series = pd.Series(end_vals, dtype=float)
    next_w = end_series[end_series.abs() > 1e-12]

    # 월말 평가: 그 시점까지 실제 체결된 매수·매도와 그날 배당반영 수정주가만 사용(보간 없음)
    ckpt_vals, ckpt_traded = {}, {}
    days_vals = days.values
    for cp in checkpoint_dates:
        k_cp = np.searchsorted(days_vals, np.datetime64(cp), side='right') - 1
        # 구간 시작 이전 체크포인트는 평가 대상에서 제외한다.
        if k_cp < 0:
            continue
        ckpt_traded[cp] = (sum(x for lst in buys.values() for x, _, kk in lst if kk <= k_cp)
                           + sum(x for lst in sells.values() for x, _, kk in lst if kk <= k_cp))
        cash_now = target_total - sum(w0.values())
        for t, lst in buys.items():
            cash_now -= sum(x for x, _, kk in lst if kk <= k_cp)
        for t in set(w0) | set(sells):
            p_start = p_start_map.get(t, np.nan)
            # 시작가격이 없거나 비정상이면 매도 현금화 계산을 할 수 없으므로 건너뛴다.
            if not (p_start == p_start and p_start > 0):
                continue
            for sq, p_sell, kk in sells.get(t, []):       # 매도 대금은 매도 시점까지의 수익을 반영한 금액으로 현금화
                # 체크포인트 이전에 유효 가격으로 체결된 매도만 현금 잔액에 반영한다.
                if kk <= k_cp and p_sell == p_sell and p_sell > 0:
                    cash_now += sq * (1 + min(p_sell / p_start - 1, WINSORIZE_CAP))
        val = cash_now
        for t in set(w0) | set(buys) | set(sells):
            p_start = p_start_map.get(t, np.nan)
            # 유효 시작가격이 없는 종목은 보유가치 수익률 평가에서 제외한다.
            if not (p_start == p_start and p_start > 0):
                continue
            keep_now = w0.get(t, 0.0) - sum(x for x, _, kk in sells.get(t, []) if kk <= k_cp)
            p_cp = px_d[t][k_cp] if t in px_d else np.nan
            # 체크포인트 당일 가격이 없으면 그 이전의 가장 최근 유효 가격을 찾는다.
            if not (p_cp == p_cp and p_cp > 0):
                arr = px_d.get(t)
                # 종목 가격 배열이 준비된 경우에만 과거 유효가격을 탐색한다.
                if arr is not None:
                    valid_idx = np.where(pd.notna(arr[:k_cp + 1]) & (arr[:k_cp + 1] > 0))[0]
                    p_cp = arr[valid_idx[-1]] if len(valid_idx) else np.nan
            # 남은 보유비중과 유효 평가가격이 모두 있을 때만 기존 포지션 가치를 더한다.
            if keep_now > 1e-12 and p_cp == p_cp and p_cp > 0:
                val += keep_now * (1 + min(p_cp / p_start - 1, WINSORIZE_CAP))
            for x, p_buy, kk in buys.get(t, []):
                # 체크포인트 전에 유효 가격으로 체결된 매수분만 현재 가치에 포함한다.
                if kk <= k_cp and p_buy == p_buy and p_buy > 0 and p_cp == p_cp and p_cp > 0:
                    val += x * (1 + min(p_cp / p_buy - 1, WINSORIZE_CAP))
        ckpt_vals[cp] = val

    return contrib_total, next_w, traded_total, achieved_total, {t: w for t, w in hold.items() if w > 1e-9}, ckpt_vals, ckpt_traded
# 3-5. 구간 수익률(비용·월말 배당반영 수정주가 평가) → 포트폴리오 → port_A·B·C 합산
# 롱·숏 레그의 체결 결과와 비용을 합쳐 한 리밸런싱 구간의 전략 수익률을 계산한다.
def strategy_window(start_date, end_date, prev_long_w, prev_short_w, cps, P):
    """리밸런싱 구간 하나(start_date~end_date)를 계산한다. 전체 백테스트와 첫 리밸런싱 예시가 모두 이 함수를 쓴다.
    월별 수익률 = (이번 체크포인트 평가액 - 직전 체크포인트 평가액) / 직전 체크포인트 평가액 (구간 시작 평가액 1.0).
    스프레드 비용은 실제로 매매한 날이 속한 달에, 대차수수료는 경과일에 비례해 부과한다."""
    n = P['n']
    long_ranked, short_ranked = select_candidates(start_date, P)
    lt = _long_weight_target(start_date, long_ranked[:n], P)
    st = _short_weight_target(start_date, short_ranked[:n], P)
    L = _liq_leg_step_ckpt(prev_long_w, long_ranked, start_date, end_date, n, 1.0, lt, cps, P['aum'], P['participation'], P['max_days'])
    S = _liq_leg_step_ckpt(prev_short_w, short_ranked, start_date, end_date, n, 1.0, st, cps, P['aum'], P['participation'], P['max_days'])
    rL, next_long_w, traded_l, achieved_l, hold_l, ckptL, cumL = L
    rS, next_short_w, traded_s, achieved_s, hold_s, ckptS, cumS = S

    rets, costs = {}, {}
    long_vals, short_vals = {}, {}   # 레그별 기여 분해용: 체크포인트별 레그 자체 가치(구간 시작=1.0, 비용 반영 전)
    prev_cpL = prev_cpS = V = 1.0
    prev_cp_date, prev_traded = start_date, 0.0
    for cp in cps:
        valL, valS = ckptL.get(cp, prev_cpL), ckptS.get(cp, prev_cpS)
        traded_to_cp = cumL.get(cp, 0.0) + cumS.get(cp, 0.0)
        elapsed = (cp - prev_cp_date).days
        cost = (traded_to_cp - prev_traded) * P['spread'] / 2 + achieved_s * P['borrow_annual'] * elapsed / 365.0 # 숏 달성비중×연 3%×경과일 단순 대차비용
        dV = (valL - prev_cpL) - (valS - prev_cpS) - cost
        rets[cp] = dV / V
        costs[cp] = cost
        long_vals[cp] = valL
        short_vals[cp] = valS
        V += dV # 성과 NAV만 누적하며 다음 구간의 기준 AUM은 키우지 않음
        prev_cpL, prev_cpS, prev_cp_date, prev_traded = valL, valS, cp, traded_to_cp

    return dict(start=start_date, end=end_date, long_ranked=long_ranked, short_ranked=short_ranked,
                long_target=lt, short_target=st, prev_long_w=prev_long_w, prev_short_w=prev_short_w,
                long_contrib=rL, short_contrib=rS, traded_long=traded_l, traded_short=traded_s,
                achieved_long=achieved_l, achieved_short=achieved_s, hold_long=hold_l, hold_short=hold_s,
                next_long_w=next_long_w, next_short_w=next_short_w,
                rets=pd.Series(rets, dtype=float), costs=pd.Series(costs, dtype=float), window_return=V - 1,
                long_vals=pd.Series(long_vals, dtype=float), short_vals=pd.Series(short_vals, dtype=float))


# 하나의 스태거드 포트폴리오를 전 기간 순차 실행하고 수익률·회전율·보유내역을 모은다.
def run_strategy_port(schedule, calendar, P, port_name='port_A', log=None):
    """포트폴리오 하나를 schedule(리밸런싱 날짜)대로 끝까지 돌린다. 반환: (월별 수익률, 리밸런싱별 회전율, 보유 내역).
    회전율은 총자산(AUM=1000억) 대비 거래금액이다: traded_long/traded_short는 이 포트폴리오 자금(P['aum']=333.3억) 대비 거래 비중이므로
    P['aum']/AUM을 곱해 전체 1000억 기준으로 환산한다(포트폴리오당 333.3억이면 대략 1/3배)."""
    rets, turnover, records = [], pd.Series(dtype=float), []
    prev_l = prev_s = pd.Series(dtype=float)
    for i in range(len(schedule) - 1):
        s, e = schedule[i], schedule[i + 1]
        W = strategy_window(s, e, prev_l, prev_s, _checkpoints(s, e, calendar), P)
        rets.append(W['rets'])
        turnover.loc[s] = (W['traded_long'] + W['traded_short']) * P['aum'] / AUM
        records += [(s, 'Long', t, w) for t, w in W['hold_long'].items()]
        records += [(s, 'Short', t, w) for t, w in W['hold_short'].items()]
        # 상세 검증 로그가 요청된 실행에서만 구간별 내부 상태를 저장한다.
        if log is not None:
            W['port'] = port_name
            log.append(W)
        prev_l, prev_s = W['next_long_w'], W['next_short_w']
    return pd.concat(rets), turnover, records


# 세 포트폴리오가 모두 운용되기 시작하는 공통 성과 측정 시작일을 찾는다.
def perf_start(schedules):
    """성과 측정 시작점: 모든 포트폴리오가 첫 편입을 시작한 날(가장 늦은 포트폴리오의 첫 리밸런싱일)."""
    return max(s[0] for s in schedules)


# port_A·B·C를 독립 실행한 뒤 월별 동일가중으로 합쳐 전체 전략 수익률을 만든다.
def run_strategy(P, schedules, calendar, log=None):
    """port_A·B·C를 각자의 리밸런싱 일정으로 독립 실행하고 월별 수익률을 평균한다.
    운용규모는 1000억 고정(세 포트폴리오 모두 항상 333.3억)이라 세 포트폴리오의 비중은 매달 같다.
    성과는 세 포트폴리오가 모두 투자된 시점(perf_start)부터 잰다. 첫 행은 그 시점(수익률 0, NAV=1)이다."""
    outs = [run_strategy_port(s, calendar, P, PORT_NAMES[k], log) for k, s in enumerate(schedules)] # A/B/C 각 고정 333.3억
    t0 = perf_start(schedules)
    ret = pd.concat([o[0] for o in outs], axis=1).mean(axis=1) # 세 고정 포트폴리오를 매월 동일가중
    ret = pd.concat([pd.Series({t0: 0.0}), ret[ret.index > t0]]).sort_index()
    ret.index.name = None
    turnover = pd.concat([o[1] for o in outs]).sort_index()
    turnover = turnover[turnover.index >= t0]
    records = [r for o in outs for r in o[2]]
    return ret, turnover, records


# 기본 전략 설정에 민감도 분석용 변경값만 덮어쓴 파라미터 묶음을 만든다.
def strategy_params(**overrides):
    """백테스트 파라미터 묶음. 기본값은 본 전략 설정이고, 민감도 분석은 overrides로 하나씩 바꾼다."""
    P = dict(factor=csi_factor, tau=TAU, n=n, cap_pct=CAP_PCT,
             long_k=LONG_WEIGHT_K, short_k=SHORT_WEIGHT_K, weight_mode='strategy',
             aum=AUM_PORT, participation=PARTICIPATION_RATE, max_days=MAX_EXECUTION_DAYS,
             spread=SPREAD, borrow_annual=BORROW_ANNUAL)
    P.update(overrides)
    return P

In [ ]:
# 분위수(롱온리) 포트폴리오: 본 전략과 같은 체결 엔진·3개 포트폴리오 스태거링·월말 배당반영 수정주가 평가. 거래정지·데이터 부족 종목은 분위 계산 전에 뺀다.
# 분위 안 종목은 동일가중, 거래비용은 왕복 스프레드만(롱온리라 대차수수료 없음).
# 지정 CSI 분위의 롱온리 포트폴리오를 한 리밸런싱 구간 동안 운용한다.
def quantile_window(start_date, end_date, prev_w, cps, q_cut, q_idx, P, bucket=None):
    """한 분위수 롱온리 포트폴리오의 한 리밸런싱 구간 수익률을 계산한다."""
    cap_today = market_cap_monthly.loc[start_date].dropna()                  # 리밸런싱일 시총 관측 종목
    universe = cap_today[cap_today >= cap_today.quantile(P['cap_pct'])].index # 시총 상위 30%
    # 보조 특성 bucket이 주어지면 먼저 해당 분위로 유니버스를 제한해 이중정렬한다.
    if bucket is not None:                       # 이중정렬: bucket = (특성 데이터, 분위 수, 분위 번호)
        b_data, b_cut, b_idx = bucket
        b_today = b_data.loc[start_date].reindex(universe).dropna()
        b_q = pd.qcut(b_today, q=b_cut, labels=False, duplicates='drop')
        universe = b_q[b_q == b_idx].index
    ff = P['factor'].loc[start_date].reindex(universe).dropna()
    ff = ff[_tradable(start_date, ff.index)] # 거래정지·가격이력 부족 종목 제외
    basket = []
    # 유효 종목 수가 분위 수 이상일 때만 qcut으로 안정적으로 포트폴리오를 구성한다.
    if len(ff) >= q_cut:
        q = pd.qcut(ff, q=q_cut, labels=False, duplicates='drop')
        basket = list(ff[q == q_idx].sort_values(kind='mergesort').index)
    rL, next_w, traded, achieved, hold, ckpt, cum = _liq_leg_step_ckpt(prev_w, basket, start_date, end_date, max(len(basket), 1), 1.0, None,
                                                                   cps, P['aum'], P['participation'], P['max_days'])
    rets, prev_v, prev_traded, V = {}, 1.0, 0.0, 1.0
    for cp in cps:
        val = ckpt.get(cp, prev_v)
        cost = (cum.get(cp, 0.0) - prev_traded) * P['spread'] / 2      # 실제 매매한 날이 속한 달에 부과
        dV = (val - prev_v) - cost
        rets[cp] = dV / V
        V += dV
        prev_v, prev_traded = val, cum.get(cp, 0.0)
    return pd.Series(rets, dtype=float), next_w


# 동일 분위 전략을 세 스태거드 일정에 적용하고 월별 동일가중 수익률로 합친다.
def run_quantile(q_cut, q_idx, P, schedules, calendar, bucket=None):
    """분위 포트폴리오를 port_A·B·C 일정별로 돌려 월별 수익률을 평균한다(본 전략과 같이 perf_start부터 측정)."""
    ports = []
    for sched in schedules:
        prev_w, rets = pd.Series(dtype=float), []
        for i in range(len(sched) - 1):
            s, e = sched[i], sched[i + 1]
            r, prev_w = quantile_window(s, e, prev_w, _checkpoints(s, e, calendar), q_cut, q_idx, P, bucket)
            rets.append(r)
        ports.append(pd.concat(rets))
    t0 = perf_start(schedules)
    ret = pd.concat(ports, axis=1).mean(axis=1)
    ret = pd.concat([pd.Series({t0: 0.0}), ret[ret.index > t0]]).sort_index()
    ret.index.name = None
    return ret

##### **본 전략 실행 (거래대금 미포함)**

전략 노트북의 헤드라인과 같은 계산이다. 이 노트북의 모든 성과·그래프는 이 `main_return` 기준이고, 6장 체결률은 이 실행의 리밸런싱 기록(`main_log`)을 그대로 쓴다.

In [ ]:
main_log = []
main_return, main_turnover, _ = run_strategy(strategy_params(), port_schedules, all_month_ends, log=main_log)
pd.DataFrame({'본 전략 (거래대금 미포함)': perf(main_return)}).round(4)

---
## **1. 분위수 검증 (Quantile Monotonicity)**

유니버스를 CSI 5분위로 나눈다(Q1 = CSI 낮음·주식수 감소 ~ Q5 = CSI 높음·주식수 증가). 각 분위를 롱온리 동일가중 포트폴리오로 만들어,
Q1에서 Q5로 갈수록 성과가 떨어지는지 본다. 본 전략과 같은 엔진(port_A·B·C 일정·거래대금 미포함·왕복 50bp·월말 배당반영 수정주가 평가)이다.

In [ ]:
q_cut = 5
quantile_df = pd.DataFrame({f"Q{i+1}": run_quantile(q_cut, i, strategy_params(), port_schedules, all_month_ends) for i in range(q_cut)})
quantile_summary = pd.DataFrame({q: perf(quantile_df[q]) for q in quantile_df}).T
quantile_summary.round(4)

In [ ]:
# Q1~Q5를 서로 구별되는 범주형 색상으로 (색맹 안전 순서: 파랑/주황/아쿠아/노랑/마젠타)
q_colors = {"Q1": "#2a78d6", "Q2": "#eb6834", "Q3": "#1baf7a", "Q4": "#eda100", "Q5": "#e87ba4"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
panels = [((1 + quantile_df.fillna(0)).cumprod(), "Cumulative NAV"),
          (np.log1p(quantile_df.fillna(0)).cumsum(), "Log Cumulative Return")]
for ax, (data, title) in zip(axes, panels):
    for q in data.columns:
        ax.plot(data.index, data[q], color=q_colors[q], linewidth=2, label=q)
    ax.set_title(f"CSI Quintile {title} (long-only, equal weight)", fontsize=10)
    ax.grid(axis="y", color="#000000", alpha=0.08)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].legend(title="CSI Quintile", frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

---
## **2. 팩터 회귀 (4-Factor)**

본 전략의 월별 수익률을 분기 격자(`quarter_ends`)로 복리 환산해 회귀한다(HAC 표준오차, 4 lag). 관측치는 2012년 2분기부터이고, 마지막 관측치는 2026-07~08의 2개월 구간이다.

- **종속변수**: 본 전략 수익률. 달러 중립 롱숏(롱 100% + 숏 100%)은 자기자금 없이 구성되는 포트폴리오라 무위험수익률(RF)을 빼지 않는다.
- **설명변수**: MKT(KOSPI − RF)·SMB·HML·MOM. 상수항 = 기존 4개 팩터로 설명되지 않는 초과수익(알파).

In [ ]:
strategy_q = _compound_to_grid(main_return.iloc[1:].rename('Strategy'), quarter_ends)
factors_q  = _compound_to_grid(factors_monthly, quarter_ends)

reg_df = pd.concat([strategy_q, factors_q], axis=1).iloc[1:]   # 첫 행(시작점, 수익률 0) 제외
y  = reg_df['Strategy']
X4 = pd.DataFrame({"MKT": reg_df['KOSPI'] - reg_df['RF'], "SMB": reg_df['SMB'], "HML": reg_df['HML'], "MOM": reg_df['MOM']})
model_4f = sm.OLS(y, sm.add_constant(X4)).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(f"표본: {reg_df.index[0].date()} ~ {reg_df.index[-1].date()} ({len(reg_df)}개 구간)")
print(model_4f.summary())

In [ ]:
# 요약 (알파 연율 = 분기 상수항 x 4)
reg_summary = {}
for k in model_4f.params.index:
    label = "alpha(상수항, 분기)" if k == "const" else k
    reg_summary[label] = model_4f.params[k]
    reg_summary[f"  t({label})"] = model_4f.tvalues[k]
reg_summary["alpha(연율, 상수항x4)"] = model_4f.params["const"] * 4
reg_summary["R2"] = model_4f.rsquared
pd.Series(reg_summary, name="4-Factor").round(4)

##### **4-Factor + CSI 팩터 (거래대금 미포함, 본 전략)**

위 4-Factor 회귀에 CSI 팩터를 설명변수로 더한다. CSI 팩터는 1장 CSI 5분위 롱온리 포트폴리오의 Q1(주식수 감소) − Q5(주식수 증가) 월수익률이다.
본 전략의 초과수익이 CSI 효과에서 나온다면 CSI 계수가 유의하고 알파는 줄어든다.

In [ ]:
csi_ls = (quantile_df['Q1'] - quantile_df['Q5']).iloc[1:].rename('CSI')   # CSI 팩터 = Q1 - Q5 (1장 분위수 결과 재사용)
csi_q = _compound_to_grid(csi_ls, quarter_ends).iloc[1:]
factors_q1 = factors_q.iloc[1:]                                            # 첫 행(시작점) 제외


# 전략 분기수익률을 4요인 또는 4요인+CSI 모형에 적합해 위험조정 알파를 검증한다.
def fit_regression(ret_m, with_csi):
    """월별 수익률을 분기 격자로 복리 환산해 4-Factor(+CSI) OLS를 적합한다(HAC 4 lag). model.summary()로 그대로 확인한다."""
    yq = _compound_to_grid(ret_m.iloc[1:].rename('y'), quarter_ends).iloc[1:]
    Xq = pd.DataFrame({"MKT": factors_q1['KOSPI'] - factors_q1['RF'], "SMB": factors_q1['SMB'],
                       "HML": factors_q1['HML'], "MOM": factors_q1['MOM']})
    # CSI의 추가 설명력을 검정하는 모형에서만 Q1-Q5 팩터를 설명변수에 포함한다.
    if with_csi:
        Xq['CSI'] = csi_q
    return sm.OLS(yq, sm.add_constant(Xq)).fit(cov_type="HAC", cov_kwds={"maxlags": 4})


model_4f_csi = fit_regression(main_return, with_csi=True)
print(f"CSI 팩터(Q1-Q5): 연율 평균 {csi_ls.mean() * 12:.2%}, 변동성 {csi_ls.std() * np.sqrt(12):.2%}, Sharpe {csi_ls.mean() / csi_ls.std() * np.sqrt(12):.2f}")
print(f"알파 연율(상수항x4): {model_4f_csi.params['const'] * 4:.2%}")
print(model_4f_csi.summary())

##### **4-Factor / 4-Factor + CSI 팩터 결과 재확인 (거래대금 미포함)**

본 전략 자체가 `PARTICIPATION_RATE=None`이므로 별도의 거래대금 반영 전략과 비교하지 않는다. 위와 동일한 `main_return`을 사용해 4-Factor와 4-Factor+CSI 회귀 결과를 다시 확인한다.

In [ ]:
nolimit_return = main_return   # 거래대금 변수를 제외한 동일 기준을 재사용

model_nolimit = fit_regression(nolimit_return, with_csi=False)
print("4-Factor (거래대금 미포함):")
print(model_nolimit.summary())

In [ ]:
model_nolimit_csi = fit_regression(nolimit_return, with_csi=True)
print(f"알파 연율(상수항x4): {model_nolimit_csi.params['const'] * 4:.2%}")
print("4-Factor + CSI (거래대금 미포함):")
print(model_nolimit_csi.summary())

**결과** (p값은 HAC 4 lag 기준, F는 HAC 공분산을 쓴 Wald F)

| | 4-Factor | 4-Factor+CSI |
|---|---|---|
| 알파(연율) | 14.4% (p=0.001) | 10.6% (p=0.008) |
| CSI 계수 | – | 0.76 (p<0.0001) |
| R² / 수정 R² | 0.10 / 0.03 | 0.34 / 0.27 |

- **CSI 팩터 계수는 0.76으로 크게 유의하다**(p<0.0001). CSI를 추가하면 수정 R²가 0.03에서 0.27로 높아져, 본 전략의 수익을 CSI 팩터가 상당 부분 설명한다.
- **알파는 줄지만 남는다**: CSI를 넣으면 연율 알파가 14.4%(p=0.001)에서 10.6%(p=0.008)로 감소하지만 5% 수준에서 여전히 유의하다.
- CSI 팩터로 설명되지 않고 남는 부분은 CSI 극단 종목 선정과 점진성·저변동성 가중의 영향일 가능성이 있으나, 이 회귀만으로 원인을 확정할 수는 없다.
- 4-Factor 모형의 수정 R²는 0.03으로 낮다. CSI를 포함한 뒤에는 전체 모형의 설명력이 개선되며, 개별 계수와 유의성은 HAC 표준오차를 기준으로 해석한다.

---
## **3. 이중정렬: 규모·모멘텀 통제 CSI 5분위 Sharpe (5x5 히트맵)**

종목 특성으로 유니버스를 먼저 5분위로 나누고, 각 분위 안에서 다시 CSI 5분위로 나눈다(25개 롱온리 동일가중 포트폴리오, 분위수 검증과 같은 엔진).
칸마다 Sharpe를 히트맵으로 그린다. 같은 특성 분위 안에서도 CSI Q1(주식수 감소)의 Sharpe가 Q5(주식수 증가)보다 꾸준히 높으면,
CSI 효과가 그 특성으로 설명되지 않는 독립적인 신호라는 근거가 된다.

| 통제 특성 | 왜 통제하나 |
|---|---|
| 시가총액 (SMB) | 증자·전환사채 발행은 소형주에 몰려 있어, CSI 효과가 소형주 효과를 다르게 잰 것일 수 있다 |
| 12-1개월 모멘텀 (MOM) | 주가가 오른 뒤 증자하고 떨어진 뒤 자사주를 사는 경향이 있어, CSI 효과가 모멘텀(반전)일 수 있다 |

2장 4-Factor 회귀에서 두 팩터 모두 5% 수준에서 유의하지 않지만, 주식수 변화 효과를 설명할 대표적인 대안 가설이라 종목 단위로도 직접 통제해 확인한다.

In [ ]:
price_m     = adj_close.resample('ME').last()
mom_monthly = price_m.shift(1) / price_m.shift(12) - 1   # 12-1개월 모멘텀

characteristics = {'SMB': (market_cap_monthly, 'Market cap', 'S'), 'MOM': (mom_monthly, 'Momentum (12-1m)', 'M')}
print("4-Factor p-value:", model_4f.pvalues[['MKT', 'SMB', 'HML', 'MOM']].round(4).to_dict())

In [ ]:
double_sort_sharpe, double_sort_cagr = {}, {}
for f, (data, label, pre) in characteristics.items():
    sh, cg = {}, {}
    for b in range(5):
        for q in range(5):
            m = perf(run_quantile(5, q, strategy_params(), port_schedules, all_month_ends, bucket=(data, 5, b)))
            sh[(f"{pre}{b+1}", f"Q{q+1}")], cg[(f"{pre}{b+1}", f"Q{q+1}")] = m['Sharpe'], m['CAGR']
    double_sort_sharpe[f], double_sort_cagr[f] = pd.Series(sh).unstack(), pd.Series(cg).unstack()

vmin = min(m.values.min() for m in double_sort_sharpe.values())
vmax = max(m.values.max() for m in double_sort_sharpe.values())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (f, mat) in zip(axes, double_sort_sharpe.items()):
    label = characteristics[f][1]
    im = ax.imshow(mat.values, cmap='Blues', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(5)); ax.set_xticklabels(mat.columns)
    ax.set_yticks(range(5)); ax.set_yticklabels([f"{r} (low)" if k == 0 else f"{r} (high)" if k == 4 else r for k, r in enumerate(mat.index)])
    ax.set_xlabel("CSI Quintile (Q1 = shares decreasing)")
    ax.set_ylabel(f"{label} quintile")
    for i in range(5):
        for j in range(5):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=9,
                    color="#ffffff" if (v - vmin) / (vmax - vmin) > 0.6 else "#1a1a1a")
    ax.set_title(f"Sharpe: {label} x CSI Quintile", fontsize=11)
fig.colorbar(im, ax=axes, shrink=0.8, label="Sharpe")
plt.show()

for f, mat in double_sort_sharpe.items():
    pre = characteristics[f][2]
    print(f"[{f}] 각 {pre}분위 안에서 Q1 - Q5 Sharpe 차이:", (mat['Q1'] - mat['Q5']).round(2).to_dict())
    print(f"     Q1 - Q5 CAGR 차이:", (double_sort_cagr[f]['Q1'] - double_sort_cagr[f]['Q5']).round(4).to_dict())

---
## **4. 파라미터 민감도**

본 전략(`main_return`)에서 파라미터 하나만 바꿔 다시 돌린다. 모든 경우 같은 엔진(port_A·B·C 일정·거래대금 미포함·월말 배당반영 수정주가 평가)과 같은 기간이다.

In [ ]:
# 기본 전략에서 지정한 설정만 변경해 민감도 분석용 수익률을 다시 계산한다.
def run_P(**overrides):
    """기본값에서 지정한 파라미터만 바꿔 전략 수익률을 다시 계산한다."""
    return run_strategy(strategy_params(**overrides), port_schedules, all_month_ends)[0]


SENS_IS_END = pd.Timestamp('2020-12-31')


# 여러 전략 결과를 전체·표본내·표본외 성과지표로 나눠 비교표를 만든다.
def sens_table_split(results):
    """전체 / 표본내(~2020-12) / 표본외(2021-01~) CAGR·Sharpe·MDD. 각 구간은 시작 시점을 NAV=1로 둔다."""
    rows = {}
    for k, r in results.items():
        oos = r[r.index >= SENS_IS_END].copy()
        oos.iloc[0] = 0.0
        f, i, o = perf(r), perf(r[r.index <= SENS_IS_END]), perf(oos)
        rows[k] = {"전체 CAGR": f['CAGR'], "전체 Sharpe": f['Sharpe'], "전체 MDD": f['MDD'],
                   "표본내 CAGR": i['CAGR'], "표본내 Sharpe": i['Sharpe'], "표본내 MDD": i['MDD'],
                   "표본외 CAGR": o['CAGR'], "표본외 Sharpe": o['Sharpe'], "표본외 MDD": o['MDD']}
    return pd.DataFrame(rows).T.round(4)

BASE = " (본 전략)"

#### **1) 종목 수(n)** — 롱·숏 각 최대 n종목

표본내(2012.03~2020.12)·표본외(2021.01~2026.08)를 나눠 보인다. n은 표본내 기준으로 정했고, 표본외 열(특히 MDD)은 참고용이다.

In [ ]:
n_results = {f"n={v}" + (BASE if v == n else ""): (main_return if v == n else run_P(n=v)) for v in [20, 30, 40, 50]}
sens_table_split(n_results)

**n=40·50에서 표본내 성과가 나쁘고 표본외에서는 그렇지 않은 이유**

표본내에는 48개월간 실제로 주식수가 줄어든 종목이 평균 28개뿐이라(40개를 넘는 달이 1%도 안 됨), n=40·50으로 올려도 롱 레그는 n=30과 사실상 같은 ~27개 종목을 그대로 담는다
(롱 레그 기여도는 n=30·40·50에서 거의 그대로다). 대신 숏 후보는 항상 충분해서, n을 올리면 신호가 약한 31~50위 종목까지 숏 바스켓에 들어오고, 이 종목들의 실제 수익률은 나쁘지 않아
숏 레그가 오히려 손실 쪽으로 기운다(숏 바스켓의 가치 변화 합이 n=30 −0.99 → n=40 −0.02 → n=50 +0.24로, 숏 레그의 수익 기여가 점점 나빠진다). 표본내 n=40·50의 성과 저하는 롱이 아니라 이 숏 레그 희석 때문이다.

표본외에는 실제 감소 종목이 평균 79개로 늘어(항상 40개 이상), n=40에서는 롱 레그가 진짜로 40개 종목으로 넓어지고 그만큼 기여도도 조금 더 좋아진다. 숏 레그 희석은 표본외에도 똑같이 있지만,
n=40에서는 이 롱 레그 개선이 숏 레그 희석을 상쇄하고도 남아 MDD가 n=30보다 오히려 좋고 Sharpe는 거의 같다(1.66 대 1.68). n=50은 실제 감소 종목이 50개에 못 미치는 달이 38%나 돼 롱 레그 개선 효과가 줄고 숏 레그 희석은 계속 커져, n=40보다 못하다.

#### **2) 가중 방식** — "동일가중"은 롱·숏 양쪽 다 동일가중을 쓴 경우

In [ ]:
weight_results = {"롱 점진성x규모 k=1.0 / 숏 저변동성 k=0.5" + BASE: main_return,
                  "롱·숏 모두 동일가중": run_P(weight_mode='equal')}
sens_table_split(weight_results)

#### **3) k값 민감도 (표본내 기준)**

k는 표본내(2012.03~2020.12) 지표만 보고 정했다. 표본외(2021.01~2026.08)를 쓰지 않고도 같은 결론이 나오는지 표본내 지표를 따로 보인다(전체 기간 지표는 참고용).
- **롱 k** (숏 k=0.5 고정): k=1.0은 최하위 종목도 0배(하한 0.02) 이상을 받아, 주식수가 줄어든 종목이 모두 담기는 최대값이다(하한에 걸리는 종목 0%).
  k를 키우면 하한에 걸려 사실상 빠지는 롱 종목이 k=2.0 11%, k=3.0 32%, k=5.0 52%로 늘어 전략 취지와 맞지 않는다. 표본내 Sharpe는 k=1.0에서 2.0까지 0.71에서 0.83으로 오르고 k=3.0(0.83)에서 멈추며 k=5.0에서는 0.70으로 떨어진다(전략 노트북 3-2).
- **숏 k** (롱 k=1.0 고정): 표본내 Sharpe만 보면 k=0이 가장 높지만 표본내 MDD는 k=0이 가장 깊다. Sharpe와 MDD를 함께 보고 k=0.5를 택했다.

In [ ]:
IS_END_K = pd.Timestamp('2020-12-31')


# 가중 강도 k별 결과에서 표본내 위험지표와 전체기간 성과를 함께 추출한다.
def perf_pair(ret):
    """k 민감도 비교용 표본내·전체 성과지표를 묶어 반환한다."""
    is_p, full_p = perf(ret[ret.index <= IS_END_K]), perf(ret)
    return {"IS Sharpe": is_p['Sharpe'], "IS MDD": is_p['MDD'],
            "전체 CAGR": full_p['CAGR'], "전체 Sharpe": full_p['Sharpe'], "전체 MDD": full_p['MDD']}


long_k_table = pd.DataFrame({f"롱 k={k}" + (BASE if k == LONG_WEIGHT_K else ""):
                             perf_pair(main_return if k == LONG_WEIGHT_K else run_P(long_k=k))
                             for k in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]}).T
short_k_table = pd.DataFrame({f"숏 k={k}" + (BASE if k == SHORT_WEIGHT_K else ""):
                              perf_pair(main_return if k == SHORT_WEIGHT_K else run_P(short_k=k))
                              for k in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]}).T
display(long_k_table.round(4))
display(short_k_table.round(4))

#### **4) 거래비용** (왕복 스프레드, 대차수수료 연 3% 고정)

In [ ]:
cost_grid = {"40bp": 0.0040, "50bp": 0.0050, "60bp": 0.0060}
cost_results = {label + (BASE if sp == SPREAD else ""): (main_return if sp == SPREAD else run_P(spread=sp)) for label, sp in cost_grid.items()}
pd.DataFrame({k: perf(v) for k, v in cost_results.items()}).T.round(4)

#### **5) 관측기간(τ)** — CSI 계산 기간과 점진성 계산 기간을 함께 바꾼다

**48개월을 고른 근거**
* **실물 경기순환 한 바퀴**: 통계청 기준순환일로 본 한국의 경기순환은 확장 평균 31개월 + 수축 평균 18개월, 한 순환이 평균 49개월이다(제1~10순환, 35~67개월.
  가장 길었던 제11순환(2013.03~2020.05, 86개월)을 넣어도 평균 약 53개월). 재고순환(키친 순환)도 약 40개월(3~4년)이다. 48개월은 경기 확장과 수축을 한 번씩 모두 담는 길이다.
* **금융시장의 발행 타이밍**: 기업은 주가·밸류에이션이 높은 호황기에 증자·전환사채를 발행하고, 주가가 낮은 불황기에 자사주를 사는 경향이 있다(market timing).
  관측기간이 경기의 한 국면(예: 12~24개월)에만 걸치면 그 국면의 일시적인 발행·매입 타이밍을 잡게 되고, 호황과 불황을 모두 거친 48개월을 봐야
  기업의 지속적인 자본정책(꾸준히 희석하는지, 꾸준히 환원하는지)이 드러난다.
* **주식수 변화가 실제로 반영되는 시간**: 국내 전환사채는 만기 3년이 일반적이고 발행 1년 뒤부터 전환할 수 있어, 발행 → 전환 → 주식수 증가가 3년 안팎에 걸쳐 나타난다.
  48개월은 이 과정을 한 번 온전히 담는다.
* **표본내 성과**: 아래 표에서 τ=48은 표본내(2012.03~2020.12) Sharpe와 MDD가 36/60/72개월보다 모두 좋다. 표본외 열은 참고용이다.

In [ ]:
tau_results = {f"τ={t}개월" + (BASE if t == TAU else ""): (main_return if t == TAU else run_P(factor=build_csi_factor(t), tau=t))
               for t in [36, 48, 60, 72]}
sens_table_split(tau_results)

#### **6) 체결 기준 확인** (왕복 50bp 고정)

거래대금 변수를 제외한 기준으로 계산한 성과를 확인한다. 종목 선정·가중·포트폴리오 일정·비용은 본 전략과 같고, 가격이 있는 종목은 목표비중을 전량 체결한다. 전체·표본내·표본외를 함께 비교한다.

In [ ]:
sens_table_split({"거래대금 미포함" + BASE: main_return})

#### **7) 시작 시점**

여러 시작 시점(2006~2014년, 성과는 각 첫 리밸런싱 2개월 뒤부터)에서 본 전략(롱 k=1.0 / 숏 k=0.5 / n=30)을 다시 돌리고, 같은 기간의 KOSPI와 비교한다. 표본내/표본외 경계는 2020-12-31로 고정한다.
CSI는 48개월 이력이 필요해 2004년부터 계산되고, 가장 이른 시작은 2006년이다.

In [ ]:
# 시작시점별 전략 성과를 같은 기간의 KOSPI 성과와 표본 구간별로 비교한다.
def vs_kospi(ret):
    """전체 / 표본내 / 표본외 각각 CSI 전략과 같은 기간 KOSPI의 CAGR·Sharpe (각 구간 시작점 NAV=1)."""
    t0_, t1_ = ret.index[0], ret.index[-1]
    k_ = factors_monthly['KOSPI']
    row = {}
    for nm, (a, b) in {"전체": (t0_, t1_), "표본내": (t0_, SENS_IS_END), "표본외": (SENS_IS_END, t1_)}.items():
        seg = lambda r: pd.concat([pd.Series({a: 0.0}), r[(r.index > a) & (r.index <= b)].dropna()])
        c, k = perf(seg(ret)), perf(seg(k_))
        row.update({(nm, "CSI CAGR"): c['CAGR'], (nm, "KOSPI CAGR"): k['CAGR'], (nm, "CSI Sharpe"): c['Sharpe'], (nm, "KOSPI Sharpe"): k['Sharpe']})
    return row


start_results = {}
for start in ['2005-12-31', '2007-12-31', '2009-12-31', '2011-12-31', '2012-12-31', '2013-12-31']:
    ame_ = pd.date_range(start=start, end=end_point, freq='ME')
    sch_ = [ame_[k::N_PORTS] for k in range(N_PORTS)]
    sch_ = [s if s[-1] == ame_[-1] else s.append(ame_[-1:]) for s in sch_]
    r_ = main_return if start == '2011-12-31' else run_strategy(strategy_params(), sch_, ame_)[0]
    start_results[f"{r_.index[0].date()}부터" + (BASE if start == '2011-12-31' else "")] = r_
start_table = pd.DataFrame({k: vs_kospi(v) for k, v in start_results.items()}).T
start_table.round(4)

모든 시작 시점에서 전체 기간 CAGR·Sharpe가 KOSPI보다 높다(CSI CAGR 12.5~17.4% 대 KOSPI 7.8~10.4%, Sharpe 0.72~1.13 대 0.33~0.44).
표본외 성과는 시작 시점과 관계없이 같다. 표본내만 떼어 봐도 모든 시작 시점에서 CSI CAGR이 KOSPI보다 높다(6.2~11.0% 대 4.0~5.6%).

#### **8) 롱 후보 선정 기준: 감소규모 순 vs 점수(감소규모×점진성) 순**

본 전략은 롱 후보를 감소규모(CSI가 낮은 순)로 최대 30개 뽑은 뒤, 그 안에서 감소규모×점진성 점수로 비중만 조정한다.
여기서는 선정 단계부터 같은 점수(감소규모×점진성, 점수가 같으면 감소규모 순)로 뽑았을 때를 비교한다. 비중 조정·숏·체결·비용은 본 전략과 같다.

In [ ]:
_select_by_csi = select_candidates


# 강건성 검정용으로 롱 후보를 CSI 감소폭 대신 감소규모×점진성 점수순으로 재정렬한다.
def select_candidates(date, P):
    """롱 후보를 점수(감소규모 x 점진성) 순으로 다시 정렬한다(점수가 같으면 감소규모 순). 숏은 그대로."""
    long_ranked, short_ranked = _select_by_csi(date, P)
    # 점수순 실험이 지정되고 순위 비교가 가능할 때만 기본 롱 순서를 대체한다.
    if P.get('long_select') == 'score' and len(long_ranked) >= 2:
        c = P['factor'].loc[date].reindex(long_ranked)
        order = pd.DataFrame({'score': (-c) * (1 - _lump_dn(date, long_ranked, P['tau'])), 'csi': c})
        long_ranked = list(order.sort_values(['score', 'csi'], ascending=[False, True], kind='mergesort').index)
    return long_ranked, short_ranked


score_select_return = run_P(long_select='score')

# 실제 감소 종목이 30개를 넘는 리밸런싱에서 두 방식의 롱 30종목 중 바뀐 종목 수
_P = strategy_params()
changed = {}
for d in all_month_ends[:-1]:
    a_, _ = _select_by_csi(d, _P)
    b_, _ = select_candidates(d, dict(_P, long_select='score'))
    # 실제 감소 후보가 편입 한도 n을 넘는 시점에서만 두 선정 방식의 차이가 발생한다.
    if len(a_) > n:
        changed[d] = len(set(a_[:n]) - set(b_[:n]))
select_candidates = _select_by_csi   # 이후 셀은 본 전략 선정 방식 그대로

print("롱 30종목 중 바뀐 종목 수 (연평균, 실제 감소 종목이 30개를 넘은 리밸런싱만):")
print(pd.Series(changed).groupby(lambda d: d.year).mean().round(1).to_dict())
sens_table_split({"감소규모 순" + BASE: main_return, "점수(감소규모x점진성) 순": score_select_return})

실제 감소 종목이 30개를 넘을 때만 뽑히는 종목이 달라진다. 후보가 적었던 2012~2019년에는 30종목 중 바뀌는 종목이 평균 0~2개로 두 방식이 사실상 같고,
후보가 많아진 2021~2022년에는 5~9개, 2023년 이후에는 10~13개가 바뀐다. 그래서 차이는 대부분 표본외에서 나온다: 점수 순 선정은 표본외 Sharpe가 1.68에서 1.82로 오르지만,
표본내에서는 Sharpe가 거의 같고(0.7061 대 0.7028) MDD는 오히려 깊다(−19.7% 대 −21.9%). 본 전략은 표본내 기준으로 판단해 감소규모 순 선정을 유지하고, 점수 순 선정은 후보가 많아진 최근 환경에서의 대안으로 참고한다.

---
## **5. 표본내 / 표본외 성과**

가중 방식(롱 k=1.0 / 숏 k=0.5)은 표본내(2012.03 ~ 2020.12) 결과만 보고 정했다. 표본외(2021.01 ~ 2026.08)는 그 결정에 쓰이지 않은 구간이다.
각 구간은 시작 시점을 NAV=1로 두고 다시 잰다(표본내 = 2012-02-29, 표본외 = 2020-12-31 기준). Sharpe는 두 시리즈 모두 초과수익(RF: factor_monthly.csv) 기준, MDD는 월말 배당반영 수정주가 평가 기준이다.

In [ ]:
IS_END = pd.Timestamp('2020-12-31')
t0, t1 = main_return.index[0], pd.Timestamp(end_point)
samples = {"표본내 (2012.03~2020.12)": (t0, IS_END), "표본외 (2021.01~2026.08)": (IS_END, t1), "전체 (2012.03~2026.08)": (t0, t1)}
kospi_m = factors_monthly['KOSPI']


# 임의 표본 구간을 시작 NAV=1로 재기준화할 수 있도록 첫 수익률 0을 붙인다.
def cut(ret, start, end):
    """(start, end] 구간의 월별 수익률 앞에 start 시점(수익률 0)을 붙여 NAV=1에서 다시 시작하게 한다."""
    s = ret[(ret.index > start) & (ret.index <= end)].dropna()
    return pd.concat([pd.Series({start: 0.0}), s])


sample_series = {"CSI 전략": main_return, "KOSPI": kospi_m}
isoos_table = pd.DataFrame({(label, name): {**perf(cut(r, s, e)), "개월 수": len(cut(r, s, e)) - 1}
                            for label, (s, e) in samples.items() for name, r in sample_series.items()}).T
isoos_table.round(4)

##### **구간별 NAV / 낙폭 (CSI 전략 vs KOSPI)**

In [ ]:
colors = {"CSI Strategy": "#2a78d6", "KOSPI": "#eb6834"}
plot_samples = {"In-sample (2012.03-2020.12)": (t0, IS_END), "Out-of-sample (2021.01-2026.08)": (IS_END, t1)}
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex='col', gridspec_kw={'height_ratios': [2.2, 1]})
for j, (title, (s, e)) in enumerate(plot_samples.items()):
    for name, r in [("CSI Strategy", main_return), ("KOSPI", kospi_m)]:
        nav = (1 + cut(r, s, e)).cumprod()
        axes[0, j].plot(nav.index, nav.values, color=colors[name], linewidth=2, label=name)
        axes[0, j].annotate(f"{nav.iloc[-1]:.2f}", (nav.index[-1], nav.iloc[-1]), xytext=(4, 0), textcoords="offset points",
                            va="center", fontsize=9, color="#333333")
        axes[1, j].plot(nav.index, (nav / nav.cummax() - 1).values, color=colors[name], linewidth=1.5)
    p = perf(cut(main_return, s, e))
    axes[0, j].set_title(f"{title}\nCSI: CAGR {p['CAGR']:.1%} | Sharpe {p['Sharpe']:.2f} | MDD {p['MDD']:.1%}", fontsize=11)
    axes[0, j].axhline(1.0, color="#000000", alpha=0.15, linewidth=1)
    axes[1, j].axhline(0.0, color="#000000", alpha=0.15, linewidth=1)
    axes[1, j].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    for ax in axes[:, j]:
        ax.grid(axis="y", color="#000000", alpha=0.08)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
axes[0, 0].set_ylabel("NAV (rebased to 1)")
axes[1, 0].set_ylabel("Drawdown")
axes[0, 0].legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

##### **구간별 분위수 성과**

1장의 CSI 5분위 롱온리 포트폴리오를 같은 두 구간으로 나눈다. 신호 자체가 표본외에서도 Q1 > Q5 순서를 유지하는지 본다.

In [ ]:
q_split = pd.DataFrame({label: {q: perf(cut(quantile_df[q], s, e))["CAGR"] for q in quantile_df} for label, (s, e) in samples.items()}).T
q_split["Q1-Q5"] = q_split["Q1"] - q_split["Q5"]
q_split.round(4)

In [ ]:
period_colors = {"In-sample": "#2a78d6", "Out-of-sample": "#eb6834"}
x = np.arange(q_cut)
width = 0.38
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, (name, label) in enumerate(zip(period_colors, list(samples)[:2])):
    vals = q_split.loc[label, [f"Q{k+1}" for k in range(q_cut)]].values
    bars = ax.bar(x + (i - 0.5) * (width + 0.02), vals, width, color=period_colors[name], label=name)
    for b, v in zip(bars, vals):
        ax.annotate(f"{v:.1%}", (b.get_x() + b.get_width() / 2, v), xytext=(0, 3 if v >= 0 else -3), textcoords="offset points",
                    ha="center", va="bottom" if v >= 0 else "top", fontsize=8, color="#333333")
ax.set_xticks(x, [f"Q{k+1}" for k in range(q_cut)])
ax.set_xlabel("CSI Quintile (Q1 = shares decreasing)")
ax.set_ylabel("CAGR")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.axhline(0, color="#000000", alpha=0.3, linewidth=1)
ax.set_title("CSI Quintile CAGR: In-sample vs Out-of-sample")
ax.grid(axis="y", color="#000000", alpha=0.08)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

---
## **6. 체결률**

본 전략이 포트폴리오(333.3억)마다 목표비중을 실제로 얼마나 채우는지, 헤드라인을 만든 리밸런싱 기록(`main_log`)으로 확인한다.

- **조건**: 거래대금 한도 없음, 편입 최대 10영업일, 가격이 있는 종목은 목표비중을 전량 체결
- **금액 체결률** = Σ min(편입 창이 끝난 뒤 보유 비중, 목표비중) ÷ 목표비중 합(=100%). 목표를 넘겨 산 몫은 세지 않는다
- **종목 체결률** = 목표비중을 다 채운 종목 수 ÷ 목표 종목 수(롱은 실제 감소 종목이 30개보다 적으면 그 수)

In [ ]:
fill_rows = []
for W in main_log:
    for leg in ('Long', 'Short'):
        lk = leg.lower()
        tgt, hold = pd.Series(W[f'{lk}_target']), pd.Series(W[f'hold_{lk}'], dtype=float)
        got = hold.reindex(tgt.index).fillna(0.0).clip(upper=tgt)
        fill_rows.append(dict(Port=W['port'], Date=W['start'], Leg=leg,
                              금액체결률=got.sum() / tgt.sum(), 종목체결률=(got >= tgt * (1 - 1e-6)).mean()))
fill_df = pd.DataFrame(fill_rows)

fill_summary = (pd.concat([fill_df, fill_df.assign(Leg='Long+Short')])
                .groupby('Leg')[['금액체결률', '종목체결률']].agg(['mean', 'min']))
print(f"리밸런싱 {len(fill_df) // 2}회(port_A·B·C 합계) x 롱/숏, {fill_df['Date'].min().date()} ~ {fill_df['Date'].max().date()}")
fill_summary.round(3)

In [ ]:
# 연도별 금액 체결률
fill_df.assign(Year=fill_df['Date'].dt.year).pivot_table(index='Leg', columns='Year', values='금액체결률', aggfunc='mean').round(3)

In [ ]:
leg_colors = {"Long": "#2a78d6", "Short": "#eb6834"}
fill_ts = fill_df.pivot_table(index='Date', columns='Leg', values='금액체결률', aggfunc='mean')
fig, ax = plt.subplots(figsize=(12, 4))
for leg in fill_ts.columns:
    ax.plot(fill_ts.index, fill_ts[leg], color=leg_colors[leg], linewidth=1.8, marker='o', markersize=3,
            label=f"{leg} (avg {fill_ts[leg].mean():.1%})")
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylabel("Fill rate (value)")
ax.set_title("Fill Rate per Rebalance: No ADV Limit, 10-day Entry Window, KRW 33.3bn per Portfolio")
ax.grid(axis="y", color="#000000", alpha=0.08)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()